# Dataset statistics

Count and group-by queries with matplotlib charts. Use this
notebook to get a feel for how the corpus is distributed across
agent types, ego actions, and environments before you go
looking for specific scenarios.

> See `01_quickstart.ipynb` for an introduction to the dataset
> and `02_query_dsl_tour.ipynb` for the query language itself.


## Setup


In [ ]:
import os
from collections import Counter
from pathlib import Path

from cascade_av.dataset import CascadeDataset
from cascade_av.query.constants import (
    ALIAS_FAMILIES,           # registry — keyed by family name
    AGENT_TYPE_PARENTS,       # vehicle / cyclist / ped / vru
    ACTION_TYPE_PARENTS,      # turn / stop_yield_decel
    ENV_TYPE_PARENTS,         # intersection / shoulder
    OBJ_TYPE_PARENTS,         # sign / debris / barrier
)

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [ ]:
ds = CascadeDataset(Path(os.environ["CASCADE_AV_DATASET_ROOT"]))
print(f"corpus loaded — {len(ds)} clips")


def entity_count(query: str) -> int:
    """Number of matching *entities* across the corpus (not clips)."""
    return len(ds.find(query).matches)


def entities_by_attr(query: str, attr: str) -> dict:
    """Bucket matching entities by a top-level attribute (e.g. ``type``).

    Like ``ds.group_by`` but counts entities rather than clips.
    """
    out = Counter()
    for m in ds.find(query).matches:
        val = getattr(m.entity, attr, None)
        if isinstance(val, list):
            for v in val:
                out[v] += 1
        else:
            out[val] += 1
    return dict(out)


def actions_with_flag(query: str, flag: str) -> int:
    """Number of matching *action* entities where ``flag`` is True.

    Use with action-scoped queries (e.g. ``agent(type=ped,
    action(jaywalk=true))``). Flag examples: ``jaywalk``, ``erratic``,
    ``aggressive``, ``illegal``, ``aborted``, ``unprotected``.
    """
    return sum(
        1 for m in ds.find(query).matches
        if getattr(m.entity, flag, False) is True
    )


# --- vocabulary-driven distribution helpers --------------------------------
# Drive every distribution chart from `cascade_av.query.constants` so new
# schema values appear automatically without editing the notebook.

_PARENT_MAPS = {
    "agent_type":  AGENT_TYPE_PARENTS,
    "action_type": ACTION_TYPE_PARENTS,
    "env_type":    ENV_TYPE_PARENTS,
    "obj_type":    OBJ_TYPE_PARENTS,
}


def leaves_for(family: str) -> list[str]:
    """Vocabulary leaves only — strips parent aliases (which are unions of leaves)."""
    parents = _PARENT_MAPS.get(family, {})
    return [k for k in ALIAS_FAMILIES[family] if k not in parents]


def parents_for(family: str) -> list[str]:
    """Parent aliases for a family (empty list for families with no hierarchy)."""
    return list(_PARENT_MAPS.get(family, {}))


def distribution(key: str, family: str) -> pd.DataFrame:
    """Clip + entity counts for every leaf of `family`, bucketed by DSL `key`.

    Examples:
        distribution("agent.type",        "agent_type")
        distribution("ego.action.type",   "action_type")
        distribution("agent.action.type", "action_type")
        distribution("env.type",          "env_type")
        distribution("obj.type",          "obj_type")
        distribution("light.color",       "light_color")
        distribution("light.state",       "light_state")
        distribution("light.shape",       "light_shape")
        distribution("cond.type",         "cond_type")
        distribution("ego.judgment",      "driving_judgment")
    """
    # Query per leaf so we drive the chart on alias labels ("car", "truck")
    # rather than raw schema strings ("oxd:Car", "PublicBus") that
    # ds.group_by exposes. Cost: 2N queries instead of 2, but N <= 24 leaves
    # and each query is a fast in-memory tree walk.
    leaves = leaves_for(family)
    clips    = {leaf: ds.count(f"{key} = {leaf}")    for leaf in leaves}
    entities = {leaf: entity_count(f"{key} = {leaf}") for leaf in leaves}
    df = pd.DataFrame({
        "clips":    pd.Series(clips,    index=leaves),
        "entities": pd.Series(entities, index=leaves),
    }).fillna(0).astype(int)
    return df.sort_values("entities", ascending=True)


def parent_rollup(key: str, family: str) -> pd.Series:
    """Clip count per parent alias (one DSL query per parent)."""
    parents = parents_for(family)
    if not parents:
        return pd.Series(dtype=int)
    return pd.Series(
        {p: ds.count(f"{key} = {p}") for p in parents},
        name="clips",
    ).sort_values(ascending=False)


def distribution_chart(
    key: str,
    family: str,
    title: str,
    *,
    color: str = "#3b6ea5",
    color_light: str = "#8aa6c8",
) -> pd.DataFrame:
    """Standard distribution chart: horizontal grouped bars (clips + entities)."""
    df = distribution(key, family)
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(df) + 1.5))
    y = range(len(df))
    ax.barh([i + 0.2 for i in y], df["clips"],    height=0.4, color=color,       label="clips")
    ax.barh([i - 0.2 for i in y], df["entities"], height=0.4, color=color_light, label="entities")
    ax.set_yticks(list(y))
    ax.set_yticklabels(df.index)
    ax.set_xlabel("count")
    ax.set_title(title)
    ax.legend(loc="lower right", frameon=False)
    plt.tight_layout()
    plt.show()
    return df


## Clips vs entities

The query API exposes two complementary perspectives:

- **`ds.count(query)`** — the number of *clips* with ≥1 match.
  Tells you about *coverage*: "how many clips even contain
  this thing?"
- **`len(ds.find(query).matches)`** — the number of *matching
  entities*. Tells you about *prevalence*: "how often does
  this thing occur?" A single clip can contribute many
  entities (six pedestrians, four vehicles, …).

Both are useful — coverage tells you how varied the corpus
is; prevalence tells you the actual instance counts.


## Headline counts

Per-namespace summary of the corpus. One row per entity family —
how many clips contain ≥1 leaf of that family, how many entities
in total, and the three leaves contributing the most entities.
Pulled from `cascade_av.query.constants.ALIAS_FAMILIES` so the
summary stays correct when the schema gains a new leaf.

In [ ]:
NAMESPACES = [
    ("agent.type",   "agent_type"),
    ("env.type",     "env_type"),
    ("obj.type",     "obj_type"),
    ("light.color",  "light_color"),
    ("cond.type",    "cond_type"),
    ("ego.judgment", "driving_judgment"),
]

rows = []
for key, family in NAMESPACES:
    df = distribution(key, family)
    leaves = " or ".join(f"{key} = {leaf}" for leaf in df.index)
    total_clips = ds.count(leaves)
    top3 = df.sort_values("entities", ascending=False).head(3).index.tolist()
    rows.append({
        "family":        family,
        "DSL key":       key,
        "leaves":        len(df),
        "clips (any)":   total_clips,
        "entities":      int(df["entities"].sum()),
        "top 3 leaves":  ", ".join(top3),
    })
headline = pd.DataFrame(rows)
headline

## Distribution of agent types

All 15 leaf kinds in the `agent_type` vocabulary. Parents
(`vehicle`, `cyclist`, `ped`, `vru`) are summarised in the small
table below the chart — each parent is the union of its leaves
and would double-count if mixed into the same bars.

In [ ]:
agent_df = distribution_chart("agent.type", "agent_type",
                              "Agent types — clip coverage vs entity prevalence")

# Parent rollup (clip-level only — entities-per-parent doesn't help here
# because a single clip can satisfy multiple leaves under the same parent).
pd.DataFrame({"parent": parent_rollup("agent.type", "agent_type")})

## Distribution of agent actions

Action vocabulary applied to **other agents** (`agent.action.type`).
All 24 leaves; parents (`turn`, `stop_yield_decel`) are summarised
below the chart.

In [ ]:
agent_action_df = distribution_chart("agent.action.type", "action_type",
                                     "Other-agent actions",
                                     color="#4f7d4a", color_light="#a8c2a5")
pd.DataFrame({"parent": parent_rollup("agent.action.type", "action_type")})

## Distribution of ego actions

Same vocabulary as agent actions but applied to **ego**
(`ego.action.type`). Compare against the previous chart to see
which actions are ego-only or agent-only in this corpus.

In [ ]:
ego_action_df = distribution_chart("ego.action.type", "action_type",
                                   "Ego actions",
                                   color="#4f7d4a", color_light="#a8c2a5")
pd.DataFrame({"parent": parent_rollup("ego.action.type", "action_type")})

## Distribution of environment types

All leaves in the `env_type` vocabulary. Parents (`intersection`,
`shoulder`) below the chart roll the specific intersection /
shoulder leaves into their family totals.

In [ ]:
env_df = distribution_chart("env.type", "env_type",
                            "Environment types",
                            color="#8a5a3b", color_light="#c2a08a")
pd.DataFrame({"parent": parent_rollup("env.type", "env_type")})

## Distribution of environmental conditions

Per-clip weather, lighting, and road-state via the `cond` entity
(`clear` / `wet` / `snowy` / `construction` / `temp_marked`).
Tells you the mix of operating conditions in the corpus.

In [ ]:
cond_df = distribution_chart("cond.type", "cond_type",
                             "Clips per environmental condition",
                             color="#6b5a8a", color_light="#b5a8c2")
cond_df

## Distribution of traffic objects

Every leaf in the `obj_type` vocabulary (signs, debris, barriers,
portable displays, etc.). Parents below the chart roll the
leaves into their three families: `sign` / `debris` / `barrier`.

In [ ]:
obj_df = distribution_chart("obj.type", "obj_type",
                            "Traffic objects",
                            color="#a5654f", color_light="#cfa898")
pd.DataFrame({"parent": parent_rollup("obj.type", "obj_type")})

## Distribution of traffic lights

Three small charts — `light.color`, `light.state`, `light.shape`
— side by side. Use them together to find specific light
configurations (e.g. flashing red arrows).

In [ ]:
import matplotlib.colors as mcolors

def _lighten(c, alpha=0.45):
    r, g, b = mcolors.to_rgb(c)
    return (r + (1 - r) * (1 - alpha),
            g + (1 - g) * (1 - alpha),
            b + (1 - b) * (1 - alpha))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (key, family, title, c) in zip(axes, [
    ("light.color", "light_color", "Color", "#a53b3b"),
    ("light.state", "light_state", "State", "#3b6ea5"),
    ("light.shape", "light_shape", "Shape", "#4f7d4a"),
]):
    df = distribution(key, family)
    y = range(len(df))
    ax.barh([i + 0.2 for i in y], df["clips"],    height=0.4, color=c,             label="clips")
    ax.barh([i - 0.2 for i in y], df["entities"], height=0.4, color=_lighten(c),   label="entities")
    ax.set_yticks(list(y))
    ax.set_yticklabels(df.index)
    ax.set_title(title)
axes[-1].legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

## Distribution of driving judgment

Ego's self-reported driving judgment per clip
(`good` / `acceptable` / `neutral` / `bad`). One value per clip
so the entities and clips bars match exactly.

In [ ]:
judgment_df = distribution_chart("ego.judgment", "driving_judgment",
                                 "Driving judgment",
                                 color="#3b6ea5", color_light="#8aa6c8")
judgment_df

## Composite questions

DSL queries compose, so you can pose questions that combine
primitives — for example, "of the clips that contain a red
light, what fraction also contain an ego stop?"


In [ ]:
def cond_row(label: str, pop_query: str, sub_query: str):
    """One row of the composite-questions table: (label, n_sub, n_pop, n_sub/n_pop)."""
    pop = ds.count(pop_query)
    sub = ds.count(sub_query)
    return (label, sub, pop, sub / max(pop, 1))


composite = pd.DataFrame(
    [
        cond_row(
            "ego stops at red light",
            "light.color = red",
            "light.color = red and ego.action = stop",
        ),
        cond_row(
            "ego stops at stop sign",
            "obj.type = stop_sign",
            "obj.type = stop_sign and ego.action.type = stop",
        ),
        cond_row(
            "ego yields/stops at yield sign",
            "obj.type = yield_sign",
            "obj.type = yield_sign and (ego.action.type = yield or ego.action.type = stop)",
        ),
        cond_row(
            "ego decel/stops at crosswalk with ped",
            "env.type = crosswalk and agent.type = ped",
            "env.type = crosswalk and agent.type = ped "
            "and (ego.action.type = decel or ego.action.type = stop)",
        ),
        cond_row(
            "ped at crosswalk (vs any ped)",
            "agent.type = ped",
            "agent.type = ped and env.type = crosswalk",
        ),
        cond_row(
            "cyclist on cycle/bike lane (vs any cyclist)",
            "agent.type = cyclist",
            "agent.type = cyclist and (env.type = cycle_lane or env.type = bike_lane)",
        ),
        cond_row(
            "ego turn at intersection",
            "env.type = intersection",
            "env.type = intersection and "
            "(ego.action.type = turn_left or ego.action.type = turn_right or ego.action.type = uturn)",
        ),
        cond_row(
            "yellow light — ego could have cleared",
            "light.color = yellow",
            "light(color = yellow, could_have_cleared = true)",
        ),
        cond_row(
            "jaywalking ped (vs any ped)",
            "agent.type = ped",
            "agent(type = ped, action(jaywalk = true))",
        ),
    ],
    columns=["question", "matching clips", "denominator (clips)", "ratio"],
)
composite.style.format({"ratio": "{:.0%}"})


## Action-flag prevalence

Agent action records carry boolean flags that mark notable
patterns: `jaywalk`, `erratic`, `aggressive`, `illegal`,
`aborted`, `unprotected`. The rate below is
`agents-with-flag-true / agents-of-that-type`. Use it to find
rare-event scenarios and which agent type contributes most.

In [ ]:
FLAGS = ["jaywalk", "erratic", "aggressive", "illegal", "aborted", "unprotected"]
AGENT_TYPES_FOR_FLAGS = ["ped", "vehicle", "cyclist"]

flag_rows = []
for atype in AGENT_TYPES_FOR_FLAGS:
    total = entity_count(f"agent.type = {atype}")
    for flag in FLAGS:
        n = entity_count(f"agent(type = {atype}, action({flag} = true))")
        flag_rows.append((atype, flag, n, total, n / max(total, 1)))

flag_df = pd.DataFrame(
    flag_rows, columns=["agent type", "flag", "matching entities", "total entities", "rate"]
)
flag_df.style.format({"rate": "{:.1%}"})

In [ ]:
pivot = flag_df.pivot(index="flag", columns="agent type", values="rate")
fig, ax = plt.subplots(figsize=(9, 4))
pivot.plot(kind="bar", ax=ax, color=["#3b6ea5", "#8a5a3b", "#4f7d4a"])
ax.set_ylabel("fraction of agents (entity-weighted)")
ax.set_xlabel("")
ax.set_title("Action-flag prevalence by agent type")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Agent × environment co-occurrence

A heatmap of clip counts for each `agent.type` × `env.type`
pairing. Use it to spot under-represented combinations
(e.g. cyclist + tunnel, pedestrian + roundabout) and the
densest scenarios.

In [ ]:
AGENTS_FOR_MATRIX = ["ped", "cyclist", "vehicle"]
ENVS_FOR_MATRIX = ["road", "intersection", "crosswalk", "cycle_lane", "roundabout", "sidewalk"]

matrix = np.zeros((len(AGENTS_FOR_MATRIX), len(ENVS_FOR_MATRIX)), dtype=int)
for i, a in enumerate(AGENTS_FOR_MATRIX):
    for j, e in enumerate(ENVS_FOR_MATRIX):
        matrix[i, j] = ds.count(f"agent.type = {a} and env.type = {e}")

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(matrix, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(ENVS_FOR_MATRIX)), labels=ENVS_FOR_MATRIX, rotation=30, ha="right")
ax.set_yticks(range(len(AGENTS_FOR_MATRIX)), labels=AGENTS_FOR_MATRIX)
ax.set_title("Clip count by agent.type × env.type")
threshold = matrix.max() / 2 if matrix.max() else 0
for i in range(len(AGENTS_FOR_MATRIX)):
    for j in range(len(ENVS_FOR_MATRIX)):
        ax.text(
            j, i, matrix[i, j],
            ha="center", va="center",
            color="white" if matrix[i, j] > threshold else "black",
            fontsize=10,
        )
fig.colorbar(im, ax=ax, shrink=0.7, label="clips")
plt.tight_layout()
plt.show()

## Scene density (agents per clip)

How crowded is a typical clip? Histogram of the agent count
in each clip's annotation tree. Pair it with the headline
counts above to filter for sparse vs dense scenarios.

In [ ]:
agent_counts = pd.Series(
    [len(seq.annotation.annotation.agents) for seq in ds],
    name="agents per clip",
)
max_count = int(agent_counts.max())
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(
    agent_counts,
    bins=range(0, max_count + 2),
    color="#3b6ea5",
    edgecolor="white",
)
ax.set_xlabel("agents per clip")
ax.set_ylabel("number of clips")
ax.set_title(
    f"Scene density — {len(ds)} clips, "
    f"median {agent_counts.median():.0f}, "
    f"mean {agent_counts.mean():.1f}, "
    f"max {max_count}"
)
plt.tight_layout()
plt.show()